# LiteALPR Full Pipeline: Train, Eval, Infer, Export

This notebook provides a complete walkthrough from setting up the environment to training, evaluating, running inference and exporting ONNX grouped by Detection (YOLOv8n-Efficient) and Recognition (SVTR26-Tiny) models.

## Setup Environment

Run these cells first to set up the repository, install dependencies, and download pre-trained weights.

In [1]:
!git clone https://github.com/vn-anhnth/LiteALPR.git

Cloning into 'LiteALPR'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 210 (delta 24), reused 28 (delta 12), pack-reused 135 (from 1)
Receiving objects: 100% (210/210), 25.15 MiB | 33.35 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [2]:
cd LiteALPR

/kaggle/working/LiteALPR


In [3]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.1/301.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4

In [4]:
!wget -O pretrained_models/det/yolov8n_efficient/best.pt https://huggingface.co/anhone3/LiteALPR/resolve/main/yolov8n_efficient/best.pt
!wget -O pretrained_models/rec/svtr26_tiny/best.pth https://huggingface.co/anhone3/LiteALPR/resolve/main/svtr26_tiny/best.pth

--2026-09-01 07:04:57--  https://huggingface.co/anhone3/LiteALPR/resolve/main/yolov8n_efficient/best.pt
Resolving huggingface.co (huggingface.co)... 18.239.50.80, 18.239.50.103, 18.239.50.16, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.80|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a5aee33c028c3a08d21ce23/62adf5c7bf1bacc398ee34e0ff3f5799dfe84dce4ed4b96eff8e8d8355f49d58?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27best.pt%3B+filename%3D%22best.pt%22%3B&X-Xet-Cas-Uid=public&user_id=public&Expires=1788249897&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE1YWVlMzNjMDI4YzNhMDhkMjFjZTIzLzYyYWRmNWM3YmYxYmFjYzM5OGVlMzRlMGZmM2Y1Nzk5ZGZlODRkY2U0ZWQ0Yjk2ZWZmOGU4ZDgzNTVmNDlkNThcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9pZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4ODI0OTg5N319fV

In [5]:
!python tools/create_lmdb_dataset.py

Creating LMDB dataset at: ./dataset/rec/lmdb_data/train
load data from ./dataset/rec/train_labels.txt: 100%|█| 5/5 [00:00<00:00, 41775.9
make dataset, save to ./dataset/rec/lmdb_data/train: 100%|█| 5/5 [00:00<00:00, 1
Created dataset with 5 samples
Creating LMDB dataset at: ./dataset/rec/lmdb_data/val
load data from ./dataset/rec/val_labels.txt: 100%|█| 2/2 [00:00<00:00, 24672.38i
make dataset, save to ./dataset/rec/lmdb_data/val: 100%|█| 2/2 [00:00<00:00, 383
Created dataset with 2 samples
Creating LMDB dataset at: ./dataset/rec/lmdb_data/test
load data from ./dataset/rec/test_labels.txt: 100%|█| 5/5 [00:00<00:00, 32164.91
make dataset, save to ./dataset/rec/lmdb_data/test: 100%|█| 4/4 [00:00<00:00, 37
Created dataset with 4 samples


## Part 1: Detection Model (YOLOv8n-Efficient)

All workflows for the License Plate Detection model.

### 1.1 Train Detection

In [6]:
!python tools/train_det.py -c configs/det/yolov8/yolov8n_efficient.yml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Transferred 523/523 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.137 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.99 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=configs/det/yolov8/yolov8n_efficient.yml, data=/kaggle/working/LiteALPR/dataset/det/data.yaml, epochs=50, time=None, patience=100, batch=256, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=output/det/yolov8n_efficient, name=train, exist_ok=False, pretrained=pretrained_models/det/yolov8n_efficient/best.pt, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cl

### 1.2 Evaluate Detection

In [7]:
!python tools/eval_det.py -m output/det/yolov8n_efficient/train/weights/best.pt

[INFO] Starting evaluation.
Ultralytics 8.3.99 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n_efficient summary (fused): 140 layers, 1,993,871 parameters, 0 gradients, 5.6 GFLOPs
val: Scanning /kaggle/working/LiteALPR/dataset/det/val/labels.cache... 4 ima
                 Class     Images  Instances      Box(P          R      mAP50  m
                   all          4          4          1          1      0.995      0.961
Speed: 0.8ms preprocess, 15.6ms inference, 0.0ms loss, 42.3ms postprocess per image
Results saved to output/det/yolov8n_efficient/val

[INFO] ========================================
[INFO] Evaluation completed!
[INFO] Precision  : 1.0000
[INFO] Recall     : 1.0000
[INFO] mAP50      : 0.9950
[INFO] mAP50-95   : 0.9611
[INFO] Results saved to: output/det/yolov8n_efficient/eval
[INFO] ========================================


### 1.3 Infer Detection
Run detection on a directory of images and save visualizations.

In [9]:
!python tools/infer_det.py -m output/det/yolov8n_efficient/train/weights/best.pt -d dataset/det/test/images --save_log

[INFO] Initializing inference on device: CUDA:0
[INFO] Loading model: output/det/yolov8n_efficient/train/weights/best.pt
[INFO] Found 5 images in dataset/det/test/images.
[INFO] Pre-loading images into memory...
[INFO] Starting warmup (10 iterations)...
[INFO] Warmup completed. Measuring FPS...

[INFO] ========================================
[INFO] Inference Test
[INFO] Device        : CUDA:0
[INFO] Total Images  : 5
[INFO] Avg Time/Img  : 11.25 ms
[INFO] FPS           : 88.89 frames/sec
[INFO] Log saved     : output/det/yolov8n_efficient/infer/infer.log
[INFO] ========================================



### 1.4 Export ONNX

In [10]:
!python tools/export_det.py -m output/det/yolov8n_efficient/train/weights/best.pt

[INFO] Initializing export on device: cuda:0
[INFO] Loading detection model: output/det/yolov8n_efficient/train/weights/best.pt
[INFO] Exporting to ONNX (imgsz=416, opset=12, FP32, dynamo=False)...
/kaggle/working/LiteALPR/tools/export_det.py:41: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.f

## Part 2: Recognition Model (SVTR26-Tiny)

All workflows for the License Plate Text Recognition model.

### 2.1 Train Recognition

In [12]:
!torchrun --nproc_per_node=1 tools/train_rec.py \
    -c configs/rec/svtr26/svtr26_tiny.yml

[2026/09/01 07:09:42] rec INFO: ----------- Config -----------
[2026/09/01 07:09:42] rec INFO: Architecture : 
[2026/09/01 07:09:42] rec INFO:     Decoder : 
[2026/09/01 07:09:42] rec INFO:         bottleneck_channels : 128
[2026/09/01 07:09:42] rec INFO:         name : EfficientRCTCDecoder
[2026/09/01 07:09:42] rec INFO:     Encoder : 
[2026/09/01 07:09:42] rec INFO:         depths : [3, 6, 3]
[2026/09/01 07:09:42] rec INFO:         dims : [64, 128, 256]
[2026/09/01 07:09:42] rec INFO:         feat2d : True
[2026/09/01 07:09:42] rec INFO:         last_stage : False
[2026/09/01 07:09:42] rec INFO:         mixer : [['Conv', 'Conv', 'Conv'], ['Conv', 'Conv', 'Conv', 'FGlobal', 'Global', 'Global'], ['Global', 'Global', 'Global']]
[2026/09/01 07:09:42] rec INFO:         name : SVTRv2LNConvTwo33
[2026/09/01 07:09:42] rec INFO:         num_heads : [2, 4, 8]
[2026/09/01 07:09:42] rec INFO:         sub_k : [[1, 1], [2, 1], [-1, -1]]
[2026/09/01 07:09:42] rec INFO:         use_pos_embed : False

### 2.2 Evaluate Recognition

In [13]:
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -m output/rec/svtr26_tiny/train/best.pth

[2026/09/01 07:11:50] rec INFO: ----------- Config -----------
[2026/09/01 07:11:50] rec INFO: Architecture : 
[2026/09/01 07:11:50] rec INFO:     Decoder : 
[2026/09/01 07:11:50] rec INFO:         bottleneck_channels : 128
[2026/09/01 07:11:50] rec INFO:         name : EfficientRCTCDecoder
[2026/09/01 07:11:50] rec INFO:     Encoder : 
[2026/09/01 07:11:50] rec INFO:         depths : [3, 6, 3]
[2026/09/01 07:11:50] rec INFO:         dims : [64, 128, 256]
[2026/09/01 07:11:50] rec INFO:         feat2d : True
[2026/09/01 07:11:50] rec INFO:         last_stage : False
[2026/09/01 07:11:50] rec INFO:         mixer : [['Conv', 'Conv', 'Conv'], ['Conv', 'Conv', 'Conv', 'FGlobal', 'Global', 'Global'], ['Global', 'Global', 'Global']]
[2026/09/01 07:11:50] rec INFO:         name : SVTRv2LNConvTwo33
[2026/09/01 07:11:50] rec INFO:         num_heads : [2, 4, 8]
[2026/09/01 07:11:50] rec INFO:         sub_k : [[1, 1], [2, 1], [-1, -1]]
[2026/09/01 07:11:50] rec INFO:         use_pos_embed : False

### 2.3 Infer Recognition
Run text recognition directly on cropped plate images.

In [14]:
!python tools/infer_rec.py -m output/rec/svtr26_tiny/train/best.pth -d dataset/rec/test --save_log

[INFO] Initializing inference on device: CUDA
[INFO] Loading model checkpoint: output/rec/svtr26_tiny/train/best.pth
[INFO] Found 5 images in dataset/rec/test.
[INFO] Pre-loading images into tensors to isolate pure model inference time...
[INFO] Starting warmup (10 iterations)...
[INFO] Warmup completed. Measuring FPS...

[INFO] ========================================
[INFO] Inference Test
[INFO] Device        : CUDA
[INFO] Total Images  : 5
[INFO] Avg Time/Img  : 7.32 ms
[INFO] FPS           : 136.66 frames/sec
[INFO] Log saved     : output/rec/train/infer/infer.log
[INFO] ========================================



### 2.4 Export ONNX

In [15]:
!python tools/export_rec.py -m output/rec/svtr26_tiny/train/best.pth --save_path output/rec/svtr26_tiny/train/best.onnx --dynamic

[INFO] Initializing export on device: CUDA
[INFO] Loading model checkpoint: output/rec/svtr26_tiny/train/best.pth
[INFO] Exporting to ONNX (input=32x128, opset=12, FP32, dynamic=True, dynamo=False)...
[INFO] Checking ONNX model...
[INFO] ONNX export successful!
[INFO] ONNX opset: 12
[INFO] ONNX model: output/rec/svtr26_tiny/train/best.onnx
